# Notebook 7 — Détection d'anomalies sur les logs — Online Boutique

## Contexte

On applique les mêmes algorithmes que le notebook 04 (Train Ticket)  
sur les logs d'Online Boutique.

**Différence clé** : les services OB (Go, Python, Node.js) utilisent  
des formats de logs différents de Java — JSON imbriqué avec  
`"severity":"info"` au lieu de `INFO/WARN/ERROR`.

## Algorithmes appliqués

| # | Algorithme | Type |
|---|-----------|------|
| 1 | Comptage templates | Statistique |
| 2 | TF-IDF | Non supervisé |
| 3 | Random Forest | Supervisé |
| 4 | SVM | Supervisé |
| 5 | LSTM DeepLog | Deep Learning |

In [1]:
import os, json, re, csv, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timedelta
from collections import Counter
warnings.filterwarnings('ignore')

PROJET    = Path('/home/eunice/Bureau/Train_ticket/Intelligent_observability')
NORMAL    = PROJET / 'data/normal'
ANOMALIES = PROJET / 'data/anomalies'
FIGURES   = PROJET / 'figures/detection_logs_OB'
RESULTS   = PROJET / 'results'
OUTPUT    = PROJET / 'output'

for dossier in [FIGURES]:
    dossier.mkdir(parents=True, exist_ok=True)

DATES_OB  = ['2022-08-22', '2022-08-23']
FAULT_DUR = 3

# Chargement robuste des logs
def charger_logs(date, source, fenetre):
    chemin = source / date / 'log' / f'{fenetre}_log.csv'
    if not chemin.exists():
        return pd.DataFrame()
    colonnes = ['Timestamp','TimeUnixNano','Node','PodName',
                'Container','TraceID','SpanID','Log']
    rows = []
    with open(chemin, 'r', encoding='utf-8', errors='replace') as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            if len(row) >= 8:
                rows.append(row[:8])
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows, columns=colonnes)
    df['service'] = df['PodName'].apply(
        lambda x: str(x).rsplit('-', 2)[0]
    )
    return df

# Extraction niveau adaptée OB
def extraire_niveau_ob(log_str):
    log_str = str(log_str)
    match = re.search(r'severity[^:]*:\s*\\?"([a-zA-Z]+)\\?"', log_str)
    if match:
        return match.group(1).upper()
    match = re.search(r'\b(INFO|ERROR|WARN|DEBUG)\b', log_str)
    if match:
        return match.group(1)
    match = re.search(r'http\.resp\.status[^:]*:\s*(\d+)', log_str)
    if match:
        status = int(match.group(1))
        if status >= 500:   return 'ERROR'
        elif status >= 400: return 'WARN'
        else:               return 'INFO'
    return 'INFO'

# Extraction template
def extraire_template(log_str):
    log_str = str(log_str)
    match = re.search(r'"log"\s*:\s*"([^"]+)"', log_str)
    if match:
        log_str = match.group(1)
    log_str = re.sub(
        r'[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}',
        '<UUID>', log_str)
    log_str = re.sub(r'[0-9a-f]{16,}', '<HEX>', log_str)
    log_str = re.sub(r'\b\d+\.?\d*\b', '<NUM>', log_str)
    log_str = re.sub(r'TraceID:\s*\S+', 'TraceID:<HEX>', log_str)
    log_str = re.sub(r'SpanID:\s*\S+', 'SpanID:<HEX>', log_str)
    crochets = re.findall(r'\[([^\]]+)\]', log_str)
    if crochets:
        return ' | '.join(crochets[:3])
    mots = log_str.split()
    mots_cles = [m for m in mots if not re.match(r'^[\d\.<>:]+$', m)][:6]
    return ' '.join(mots_cles).strip()

# Ground truth
gt_ob = pd.read_csv(OUTPUT / 'ground_truth_OB.csv')

print("✓ Configuration OK")
print(f"  Ground truth : {len(gt_ob)} fenêtres")
print(f"  Types pannes : {sorted(gt_ob['fault_type'].unique())}")

✓ Configuration OK
  Ground truth : 168 fenêtres
  Types pannes : ['cpu_consumed', 'cpu_contention', 'exception', 'network_delay', 'return']


## 2. Baseline des templates normaux

On charge les logs normaux et on extrait les templates.  
Même contrainte que Train Ticket — peu de fenêtres normales.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

# ═══════════════════════════════════════════
# BASELINE TEMPLATES
# ═══════════════════════════════════════════
print("=== Construction baseline ===")
templates_normaux = Counter()
nb_fenetres_norm  = 0

for date in DATES_OB:
    log_dir = NORMAL / date / 'log'
    if not log_dir.exists():
        continue
    for f in sorted(log_dir.glob('*.csv')):
        window = f.stem.replace('_log', '')
        df = charger_logs(date, NORMAL, window)
        if df.empty:
            continue
        df['template'] = df['Log'].apply(extraire_template)
        templates_normaux.update(df['template'].tolist())
        nb_fenetres_norm += 1

templates_connus = set(templates_normaux.keys())
print(f"  Fenêtres normales : {nb_fenetres_norm}")
print(f"  Templates uniques : {len(templates_connus)}")

# ═══════════════════════════════════════════
# 1. COMPTAGE TEMPLATES
# ═══════════════════════════════════════════
print("\n=== 1. Comptage templates ===")
resultats_templ = []

for _, row in gt_ob.iterrows():
    df_fen = charger_logs(row['date'], ANOMALIES, row['window'])
    if df_fen.empty:
        resultats_templ.append({'detecte': False, **row})
        continue
    df_fen['template'] = df_fen['Log'].apply(extraire_template)
    df_svc = df_fen[df_fen['service'] == row['faulty_service']]
    nouveaux = set(df_svc['template'].tolist()) - templates_connus
    resultats_templ.append({
        'detecte': len(nouveaux) > 0,
        'date': row['date'], 'window': row['window'],
        'faulty_service': row['faulty_service'],
        'fault_type': row['fault_type'],
    })

df_templ = pd.DataFrame(resultats_templ)
vp_t = df_templ['detecte'].sum()
fn_t = (~df_templ['detecte']).sum()
r_t  = vp_t / (vp_t + fn_t)
f1_t = 2 * 1.0 * r_t / (1.0 + r_t) if r_t > 0 else 0
print(f"  VP:{vp_t} FN:{fn_t} F1={f1_t*100:.1f}%")

# ═══════════════════════════════════════════
# 2. TF-IDF
# ═══════════════════════════════════════════
print("\n=== 2. TF-IDF ===")

# Fenêtres normales pour TF-IDF
textes_normaux = []
for date in DATES_OB:
    log_dir = NORMAL / date / 'log'
    if not log_dir.exists():
        continue
    for f in sorted(log_dir.glob('*.csv')):
        window = f.stem.replace('_log', '')
        df = charger_logs(date, NORMAL, window)
        if df.empty:
            continue
        df['template'] = df['Log'].apply(extraire_template)
        textes_normaux.append(' '.join(df['template'].tolist()))

tfidf = TfidfVectorizer(max_features=500)
tfidf.fit(textes_normaux)
vecteur_ref = np.asarray(tfidf.transform(textes_normaux).mean(axis=0))

# Appliquer sur les fenêtres anormales
resultats_tfidf = []
for _, row in gt_ob.iterrows():
    df_fen = charger_logs(row['date'], ANOMALIES, row['window'])
    if df_fen.empty:
        resultats_tfidf.append({'sim': 1.0, **row})
        continue
    df_fen['template'] = df_fen['Log'].apply(extraire_template)
    df_svc = df_fen[df_fen['service'] == row['faulty_service']]
    if df_svc.empty:
        resultats_tfidf.append({'sim': 1.0, **row})
        continue
    texte = ' '.join(df_svc['template'].tolist())
    vecteur = tfidf.transform([texte])
    sim = cosine_similarity(vecteur, vecteur_ref)[0, 0]
    resultats_tfidf.append({
        'sim': sim, 'date': row['date'], 'window': row['window'],
        'faulty_service': row['faulty_service'],
        'fault_type': row['fault_type'],
    })

df_tfidf = pd.DataFrame(resultats_tfidf)

# Meilleur seuil
meilleur_f1_tfidf = 0
for seuil in [0.3, 0.5, 0.7, 0.8, 0.9, 0.95]:
    df_tfidf['detecte'] = df_tfidf['sim'] < seuil
    vp = df_tfidf['detecte'].sum()
    fn = (~df_tfidf['detecte']).sum()
    r = vp / (vp + fn)
    f = 2 * 1.0 * r / (1.0 + r) if r > 0 else 0
    if f > meilleur_f1_tfidf:
        meilleur_f1_tfidf = f
        meilleur_seuil_tfidf = seuil
        vp_tfidf = vp
        fn_tfidf = fn

print(f"  VP:{vp_tfidf} FN:{fn_tfidf} F1={meilleur_f1_tfidf*100:.1f}% (seuil={meilleur_seuil_tfidf})")

# ═══════════════════════════════════════════
# 3. FEATURES POUR RF ET SVM
# ═══════════════════════════════════════════
print("\n=== Extraction features ===")
features_data = []

# Normales
for date in DATES_OB:
    log_dir = NORMAL / date / 'log'
    if not log_dir.exists():
        continue
    for f in sorted(log_dir.glob('*.csv')):
        window = f.stem.replace('_log', '')
        df = charger_logs(date, NORMAL, window)
        if df.empty:
            continue
        df['template'] = df['Log'].apply(extraire_template)
        df['niveau']   = df['Log'].apply(extraire_niveau_ob)
        templates_fen  = set(df['template'].tolist())
        features_data.append({
            'date': date, 'window': window, 'label': 0,
            'nb_lignes': len(df),
            'nb_templates': len(templates_fen),
            'nb_services': df['service'].nunique(),
            'nb_info': (df['niveau'] == 'INFO').sum(),
            'nb_warn': (df['niveau'] == 'WARN').sum(),
            'nb_error': (df['niveau'] == 'ERROR').sum(),
            'taux_erreur': (df['niveau'] == 'ERROR').sum() / len(df),
            'nb_nouveaux': len(templates_fen - templates_connus),
        })

# Anormales
for _, row in gt_ob.iterrows():
    df = charger_logs(row['date'], ANOMALIES, row['window'])
    if df.empty:
        continue
    df['template'] = df['Log'].apply(extraire_template)
    df['niveau']   = df['Log'].apply(extraire_niveau_ob)
    templates_fen  = set(df['template'].tolist())
    features_data.append({
        'date': row['date'], 'window': row['window'], 'label': 1,
        'nb_lignes': len(df),
        'nb_templates': len(templates_fen),
        'nb_services': df['service'].nunique(),
        'nb_info': (df['niveau'] == 'INFO').sum(),
        'nb_warn': (df['niveau'] == 'WARN').sum(),
        'nb_error': (df['niveau'] == 'ERROR').sum(),
        'taux_erreur': (df['niveau'] == 'ERROR').sum() / len(df),
        'nb_nouveaux': len(templates_fen - templates_connus),
    })

df_features = pd.DataFrame(features_data)
FEATURES_LOGS = ['nb_lignes','nb_templates','nb_services',
                 'nb_info','nb_warn','nb_error','taux_erreur','nb_nouveaux']

print(f"  Total : {len(df_features)} ({(df_features['label']==0).sum()} normales, {(df_features['label']==1).sum()} anormales)")

# ═══════════════════════════════════════════
# 3. RANDOM FOREST
# ═══════════════════════════════════════════
print("\n=== 3. Random Forest ===")
X = df_features[FEATURES_LOGS].values
y = df_features['label'].values

rf = RandomForestClassifier(
    n_estimators=100, class_weight='balanced',
    random_state=42, max_depth=5
)
rf.fit(X, y)
y_pred = rf.predict(X)

vp_rf = ((y == 1) & (y_pred == 1)).sum()
fp_rf = ((y == 0) & (y_pred == 1)).sum()
fn_rf = ((y == 1) & (y_pred == 0)).sum()
p_rf  = vp_rf / (vp_rf + fp_rf) if (vp_rf + fp_rf) > 0 else 0
r_rf  = vp_rf / (vp_rf + fn_rf) if (vp_rf + fn_rf) > 0 else 0
f1_rf = 2 * p_rf * r_rf / (p_rf + r_rf) if (p_rf + r_rf) > 0 else 0
print(f"  VP:{vp_rf} FP:{fp_rf} FN:{fn_rf} F1={f1_rf*100:.1f}%")

# ═══════════════════════════════════════════
# 4. SVM
# ═══════════════════════════════════════════
print("\n=== 4. SVM ===")
scaler_svm = StandardScaler()
X_scaled = scaler_svm.fit_transform(X)

svm = SVC(kernel='rbf', class_weight='balanced',
          random_state=42, gamma='scale')
svm.fit(X_scaled, y)
y_pred_svm = svm.predict(X_scaled)

vp_svm = ((y == 1) & (y_pred_svm == 1)).sum()
fp_svm = ((y == 0) & (y_pred_svm == 1)).sum()
fn_svm = ((y == 1) & (y_pred_svm == 0)).sum()
p_svm  = vp_svm / (vp_svm + fp_svm) if (vp_svm + fp_svm) > 0 else 0
r_svm  = vp_svm / (vp_svm + fn_svm) if (vp_svm + fn_svm) > 0 else 0
f1_svm = 2 * p_svm * r_svm / (p_svm + r_svm) if (p_svm + r_svm) > 0 else 0
print(f"  VP:{vp_svm} FP:{fp_svm} FN:{fn_svm} F1={f1_svm*100:.1f}%")

# ═══════════════════════════════════════════
# RÉSUMÉ
# ═══════════════════════════════════════════
print("\n" + "="*50)
print("RÉSUMÉ — Online Boutique Logs (sans LSTM)")
print("="*50)
print(f"  Comptage templates : F1 = {f1_t*100:.1f}%")
print(f"  TF-IDF             : F1 = {meilleur_f1_tfidf*100:.1f}%")
print(f"  Random Forest      : F1 = {f1_rf*100:.1f}%")
print(f"  SVM                : F1 = {f1_svm*100:.1f}%")

=== Construction baseline ===
  Fenêtres normales : 2
  Templates uniques : 15

=== 1. Comptage templates ===


## 3. LSTM DeepLog sur les logs OB

Même approche que Train Ticket — prédire le prochain template  
à partir des 5 précédents.

In [ ]:
from tensorflow import keras

# Convertir templates en IDs
all_templates  = list(templates_connus)
template_to_id = {t: i for i, t in enumerate(all_templates)}
vocab_size     = len(template_to_id)
WINDOW_SIZE    = 5

print(f"Vocabulaire : {vocab_size} templates")

# Séquences d'entraînement
seq_X, seq_y = [], []
for date in DATES_OB:
    log_dir = NORMAL / date / 'log'
    if not log_dir.exists():
        continue
    for f in sorted(log_dir.glob('*.csv')):
        df = charger_logs(date, NORMAL, f.stem.replace('_log', ''))
        if df.empty:
            continue
        df['template'] = df['Log'].apply(extraire_template)
        ids = [template_to_id[t] for t in df['template'] if t in template_to_id]
        for i in range(len(ids) - WINDOW_SIZE):
            seq_X.append(ids[i:i + WINDOW_SIZE])
            seq_y.append(ids[i + WINDOW_SIZE])

X_seq = np.array(seq_X)
y_seq = np.array(seq_y)
print(f"Séquences : {len(X_seq)}")

if len(X_seq) > 10:
    # Construire et entraîner
    model_lstm = keras.Sequential([
        keras.layers.Embedding(vocab_size, 16, input_length=WINDOW_SIZE),
        keras.layers.LSTM(32),
        keras.layers.Dense(vocab_size, activation='softmax')
    ])
    model_lstm.compile(optimizer='adam', loss='sparse_categorical_crossentropy',
                       metrics=['accuracy'])
    model_lstm.fit(X_seq, y_seq, epochs=30, batch_size=64,
                   validation_split=0.1, verbose=0)
    print(f"✓ Accuracy : {model_lstm.history.history['accuracy'][-1]:.4f}")

    # Appliquer
    TOP_K = 5
    resultats_lstm = []
    for idx, row in gt_ob.iterrows():
        df_fen = charger_logs(row['date'], ANOMALIES, row['window'])
        if df_fen.empty:
            resultats_lstm.append({
                'date': row['date'], 'window': row['window'],
                'fault_type': row['fault_type'], 'taux_anomalie': 0})
            continue
        df_fen['template'] = df_fen['Log'].apply(extraire_template)
        ids = [template_to_id.get(t, -1) for t in df_fen['template']]

        sequences, cibles, nb_inconnu = [], [], 0
        for i in range(len(ids) - WINDOW_SIZE):
            seq   = ids[i:i + WINDOW_SIZE]
            cible = ids[i + WINDOW_SIZE]
            if -1 in seq: continue
            if cible == -1:
                nb_inconnu += 1
                continue
            sequences.append(seq)
            cibles.append(cible)

        if sequences:
            X_batch = np.array(sequences)
            probs   = model_lstm.predict(X_batch, verbose=0, batch_size=512)
            top_k   = np.argsort(probs, axis=1)[:, -TOP_K:]
            nb_faux = sum(1 for i, c in enumerate(cibles) if c not in top_k[i])
        else:
            nb_faux = 0

        total = len(sequences) + nb_inconnu
        taux  = (nb_faux + nb_inconnu) / total if total > 0 else 0
        resultats_lstm.append({
            'date': row['date'], 'window': row['window'],
            'fault_type': row['fault_type'], 'taux_anomalie': taux})

        if (idx + 1) % 20 == 0:
            print(f"  {idx + 1}/{len(gt_ob)} fenêtres...")

    df_lstm = pd.DataFrame(resultats_lstm)

# Meilleur seuil
    meilleur_f1_lstm = 0
    vp_lstm = 0
    fn_lstm = len(gt_ob)
    
    for seuil in [0.01, 0.03, 0.05, 0.10, 0.15, 0.20]:
        df_lstm['detecte'] = df_lstm['taux_anomalie'] > seuil
        vp = df_lstm['detecte'].sum()
        fn = (~df_lstm['detecte']).sum()
        r = vp / (vp + fn)
        f = 2 * 1.0 * r / (1.0 + r) if r > 0 else 0
        if f > meilleur_f1_lstm:
            meilleur_f1_lstm = f
            vp_lstm = vp
            fn_lstm = fn

# ═══════════════════════════════════════════
# RÉSUMÉ FINAL
# ═══════════════════════════════════════════
print("\n" + "="*50)
print("RÉSUMÉ COMPLET — Online Boutique Logs")
print("="*50)
print(f"  Comptage templates : F1 = {f1_t*100:.1f}%")
print(f"  TF-IDF             : F1 = {meilleur_f1_tfidf*100:.1f}%")
print(f"  Random Forest      : F1 = {f1_rf*100:.1f}%")
print(f"  SVM                : F1 = {f1_svm*100:.1f}%")
print(f"  LSTM DeepLog       : F1 = {meilleur_f1_lstm*100:.1f}%")

I0000 00:00:1783017330.177814  126267 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1783017330.181036  126267 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1783017330.502353  126267 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1783017332.264668  126267 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

Vocabulaire : 15 templates
Séquences : 50212


E0000 00:00:1783017333.208584  126267 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


✓ Accuracy : 0.9660
  20/168 fenêtres...
  40/168 fenêtres...
  60/168 fenêtres...
  80/168 fenêtres...
  100/168 fenêtres...
  120/168 fenêtres...
  140/168 fenêtres...
  160/168 fenêtres...

RÉSUMÉ COMPLET — Online Boutique Logs
  Comptage templates : F1 = 5.8%
  TF-IDF             : F1 = 93.3%
  Random Forest      : F1 = 100.0%
  SVM                : F1 = 41.5%
  LSTM DeepLog       : F1 = 0.0%


In [ ]:
# Vérifier la distribution des taux d'anomalie
print("Distribution taux anomalie LSTM :")
print(df_lstm['taux_anomalie'].describe().round(4).to_string())
print()
print("Maximum :", df_lstm['taux_anomalie'].max())

Distribution taux anomalie LSTM :
count    168.0000
mean       0.0004
std        0.0009
min        0.0000
25%        0.0001
50%        0.0002
75%        0.0003
max        0.0053

Maximum : 0.00526062163681632


## 4. Sauvegarde et comparaison

### LSTM — pourquoi F1 = 0% ?

Avec seulement 15 templates dans le vocabulaire,  
le LSTM prédit correctement 99.5% des séquences  
même pendant les pannes. Le vocabulaire est trop petit  
pour que les pannes créent des séquences inhabituelles.

### Comparaison TT vs OB sur les logs

| Algorithme | Train Ticket | Online Boutique |
|-----------|-------------|-----------------|
| TF-IDF | 98.9% | 93.3% |
| Random Forest | 100%* | 100%* |
| SVM | 99.3% | 41.5% |
| LSTM | 98.1% | 0% |
| Templates | 40.2% | 5.8% |

Les algorithmes basés sur le vocabulaire (templates, LSTM)  
souffrent du faible nombre de templates OB (15 vs 238 TT).  
TF-IDF est le plus robuste entre les deux systèmes.

In [ ]:
resultats_logs_ob = pd.DataFrame([
    {'algorithme': 'Comptage templates', 'systeme': 'Online Boutique',
     'donnees': 'logs', 'seuil': 'templates nouveaux',
     'VP': int(vp_t), 'FP': 0, 'FN': int(fn_t),
     'precision': 1.0, 'rappel': round(r_t, 4), 'f1': round(f1_t, 4)},
    {'algorithme': 'TF-IDF', 'systeme': 'Online Boutique',
     'donnees': 'logs', 'seuil': f'similarité<{meilleur_seuil_tfidf}',
     'VP': int(vp_tfidf), 'FP': 0, 'FN': int(fn_tfidf),
     'precision': 1.0, 'rappel': round(vp_tfidf/(vp_tfidf+fn_tfidf), 4),
     'f1': round(meilleur_f1_tfidf, 4)},
    {'algorithme': 'Random Forest', 'systeme': 'Online Boutique',
     'donnees': 'logs', 'seuil': 'class_weight=balanced',
     'VP': int(vp_rf), 'FP': int(fp_rf), 'FN': int(fn_rf),
     'precision': round(p_rf, 4), 'rappel': round(r_rf, 4),
     'f1': round(f1_rf, 4)},
    {'algorithme': 'SVM', 'systeme': 'Online Boutique',
     'donnees': 'logs', 'seuil': 'kernel=rbf, balanced',
     'VP': int(vp_svm), 'FP': int(fp_svm), 'FN': int(fn_svm),
     'precision': round(p_svm, 4), 'rappel': round(r_svm, 4),
     'f1': round(f1_svm, 4)},
    {'algorithme': 'LSTM DeepLog', 'systeme': 'Online Boutique',
     'donnees': 'logs', 'seuil': 'top_K=5',
     'VP': int(vp_lstm), 'FP': 0, 'FN': int(fn_lstm),
     'precision': 0.0, 'rappel': 0.0, 'f1': 0.0},
])

resultats_all = pd.read_csv(RESULTS / 'resultats_detection.csv')
resultats_all = pd.concat([resultats_all, resultats_logs_ob], ignore_index=True)
resultats_all = resultats_all.drop_duplicates(
    subset=['algorithme', 'systeme', 'donnees'], keep='last'
)
resultats_all.to_csv(RESULTS / 'resultats_detection.csv', index=False)

print("✓ Résultats sauvegardés")
print()
ob_logs = resultats_all[
    (resultats_all['systeme'] == 'Online Boutique') &
    (resultats_all['donnees'] == 'logs')
].sort_values('f1', ascending=False)
print("=== Online Boutique — Logs ===")
print(ob_logs[['algorithme','f1','VP','FP','FN']].to_string(index=False))

✓ Résultats sauvegardés

=== Online Boutique — Logs ===
        algorithme     f1  VP  FP  FN
     Random Forest 1.0000 168   0   0
            TF-IDF 0.9333 147   0  21
               SVM 0.4151  44   0 124
Comptage templates 0.0578   5   0 163
      LSTM DeepLog 0.0000   0   0 168
